# Face Mask Detection — Colab Training

Runs all three experiments on a Colab GPU and exports `results/` back to Google Drive so you can download it to your laptop.

## Before running
1. **Runtime → Change runtime type → T4 GPU**
2. In your Google Drive, create a folder `FaceMask/` and upload:
   - The project **code** — either a zip of `src/` + `run_all.py` + `requirements.txt`, or clone from GitHub in the cell below.
   - The **dataset** as a zip: `dataset_processed.zip` (containing `Train/`, `Validation/`, `Test/`).

Expected Drive layout:
```
MyDrive/FaceMask/
├── project.zip          # contains src/, run_all.py
└── dataset_processed.zip
```

## 1. Verify GPU

In [ ]:
!nvidia-smi
import tensorflow as tf
print('TF', tf.__version__, 'GPUs:', tf.config.list_physical_devices('GPU'))

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Load code + dataset

Pick **ONE** of the two approaches below and run only that cell.

### Option A: From zip in Drive (recommended)

In [ ]:
import os, shutil
DRIVE_DIR = '/content/drive/MyDrive/FaceMask'
WORK      = '/content/project'
DATA      = '/content/dataset_processed'

os.makedirs(WORK, exist_ok=True)
!unzip -q -o "$DRIVE_DIR/project.zip"            -d "$WORK"
!unzip -q -o "$DRIVE_DIR/dataset_processed.zip"  -d /content

# Flatten project dir in case zip had a top-level folder
entries = [e for e in os.listdir(WORK) if not e.startswith('.')]
if len(entries) == 1 and os.path.isdir(os.path.join(WORK, entries[0])):
    inner = os.path.join(WORK, entries[0])
    for e in os.listdir(inner):
        shutil.move(os.path.join(inner, e), WORK)
    os.rmdir(inner)

!ls "$WORK"
!ls "$DATA" | head

### Option B: From GitHub repo

In [ ]:
# Replace <YOUR_USER>/<YOUR_REPO> and unzip the dataset from Drive.
# WORK = '/content/project'
# DATA = '/content/dataset_processed'
# !git clone https://github.com/<YOUR_USER>/<YOUR_REPO>.git $WORK
# !unzip -q -o '/content/drive/MyDrive/FaceMask/dataset_processed.zip' -d /content

## 4. Install extra deps (most are pre-installed on Colab)

In [ ]:
!pip install -q opencv-python seaborn scikit-learn
# tensorflow_hub is needed only for the TF-Hub ViT; skip if you pass --skip_vit
!pip install -q tensorflow-hub

## 5. Train everything

`--fast` enables mixed_float16 + XLA on the Colab GPU. Expect ~30–60 min on a T4.

In [ ]:
%cd /content/project
!python run_all.py --data_dir /content/dataset_processed --fast

### (Optional) run a single experiment only

In [ ]:
# !python run_all.py --only 2 --data_dir /content/dataset_processed --fast
# !python run_all.py --only 3 --skip_vit --data_dir /content/dataset_processed --fast

## 6. Export results back to Drive

This copies everything in `results/` to `MyDrive/FaceMask/results_<timestamp>/` so you can download it to your laptop from Drive.

In [ ]:
import os, shutil, datetime
stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M')
src   = '/content/project/results'
dst   = f'/content/drive/MyDrive/FaceMask/results_{stamp}'
shutil.copytree(src, dst)
print('Exported ->', dst)

# Also create a single zip for easy download
zip_path = f'/content/drive/MyDrive/FaceMask/results_{stamp}.zip'
shutil.make_archive(zip_path.replace('.zip', ''), 'zip', src)
print('Zipped   ->', zip_path)

## 7. (Optional) download results directly to this browser

In [ ]:
from google.colab import files
files.download(zip_path)